# Huấn luyện model riêng cho AI Teacher

Notebook này huấn luyện một phần bổ sung (LoRA) cho Qwen 3.5 9B, dùng dữ liệu đã chuẩn bị sẵn.

**Cách dùng:**
1. Vào menu *Runtime > Change runtime type* và chọn **T4 GPU**.
2. Bấm chạy lần lượt từng ô từ trên xuống. Ô 3 sẽ hỏi tệp: chọn `colab-package.zip`.
3. Ô 5 chạy lâu nhất (ước tính 30 đến 60 phút). Giữ tab này mở.
4. Ô 6 tải `lora.zip` về máy. Đưa tệp đó lại cho Claude để gộp vào app và đo chất lượng.

Notebook không chứa khóa nào. Bạn tự đăng nhập Google; không nhập mật khẩu vào ô nào ở đây.

In [ ]:
# 1. Kiểm tra đã có GPU chưa (phải thấy dòng có chữ T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Cài đúng phiên bản
!pip install -q transformers==5.22.2 peft==0.21.2 accelerate==1.15.0 safetensors bitsandbytes==0.50.2 flash-linear-attention==0.6.0
!pip install -q causal-conv1d==1.7.0 || echo "causal-conv1d did not build: training is still correct, only slower"

In [ ]:
# 3. Tải lên colab-package.zip rồi giải nén
from google.colab import files
import zipfile, pathlib

uploaded = files.upload()
name = next(iter(uploaded))
pathlib.Path('/content/work').mkdir(exist_ok=True)
zipfile.ZipFile(name).extractall('/content/work')
print(sorted(p.name for p in pathlib.Path('/content/work').iterdir()))

In [ ]:
# 4. Chỉ chạy ô này nếu ô 5 báo lỗi 401 hoặc 403 (cần đăng nhập HuggingFace)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 5. Huấn luyện
%cd /content/work
!python train_teacher.py --data data --out lora

In [ ]:
# 6. Tải kết quả về máy
import shutil
from google.colab import files

shutil.make_archive('/content/lora', 'zip', '/content/work/lora')
files.download('/content/lora.zip')